In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

RANDOM_STATE = 5

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
def evaluate(preds, y, target_names, title=""):
    """Print report + F2, show confusion matrix. Returns the predictions."""
    print(f"--- {title} ---")
    print(f"accuracy: {(preds == y).mean():.4f}")
    print(classification_report(y, preds, target_names=target_names))
    #cm = confusion_matrix(y, preds) #labels=target_names)
    #ConfusionMatrixDisplay(cm).plot()
    #return preds

# Data Preparation

In [ ]:
import glob
files = glob.glob("data/*.csv")

### Loading and Cleaning
First we must concatenate all 8 csv files. They have odd column names with trailing spaces, so names must be normalised. We then clean any rows that contain infinity or NaNs.

In [ ]:
df = (
    pd.concat((pd.read_csv(f, encoding_errors="replace") for f in files), ignore_index=True)
    .rename(columns=lambda s: s.strip())
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

### Down-Casting
Down-casting from float64 is inherantly lossy, so below checks if loss > 1e-07 when casting to float32. 

In [ ]:
for col in df.select_dtypes("float").columns:
    orig = df[col].astype("float64")
    back = df[col].astype("float32").astype("float64")
    
    # largest relative error introduced
    rel_err = ((back - orig).abs() / orig.abs().replace(0, np.nan)).max()
    print(col, rel_err)

In [ ]:
# Safely downcasts 64-bit data to 32-bit where there is no information loss
def down_cast(d):
    original_size = d.memory_usage(deep=True).sum()
    for col in d.select_dtypes("integer").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")
    for col in d.select_dtypes("float").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")
    final_size = d.memory_usage(deep=True).sum()
    print("Size savings: " + str(original_size-final_size) + " bytes")
    return d

df = down_cast(df)

We save >0.88 GB for our precious, precious RAM. 

### De-duplication

In [ ]:
original_size = df.shape[0]
df = df.drop_duplicates()
new_size = df.shape[0]
original_size-new_size

CIC-IDS2017 contains a lot of duplicate rows (307,084!!). These must be removed to prevent leakage - a row could be in the training set and then it's twin could appear in the test set. Now we definitely don't want that! It would produce lovely metrics but a terrible model.

### Normalising non utf-8 characters

In [ ]:
import unicodedata

def normalise(s):
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore")
            .decode("ascii"))

X = df.drop(["Label", "Destination Port"], axis=1)
y = df["Label"].apply(normalise)

In [ ]:
y.value_counts()

### Data splitting

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42)
X_test, X_cv, y_test, y_cv       = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42)

In [ ]:
X_train.shape[0] + X_test.shape[0] + X_cv.shape[0] == df.shape[0]

### Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
std_scaler = StandardScaler().fit(X_train)
X_train = std_scaler.transform(X_train)
X_cv    = std_scaler.transform(X_cv)
X_test  = std_scaler.transform(X_test)

In [ ]:
X_train.shape

# Imbalance experiment
This dataset is super imbalanced. There are >2 million benign flows, and some attack categories are in the hundreds of thousands. Wheras Heartbleed, Sql injection, and Infiltration attack classes have less than 100 examples.
We will compare three methods for handling imbalance:
- Modifying class weights;
- Undersampling the majority;
- SMOTE.

These will be evaluated mostly factoring macro-recall, as our aim is to minimise missed attacks at the cost of reduced precision (an increase in false positives).

This will be a fair test, controlling for:
- train, test and CV split (these sets will be constant);
- scaling and other pre-processing;
- choose of model (XGBoost);
- hyper-parameters. To reduce computation cost while maintaining a fair comparison, all experiments will use identical hyper-parameters with a learning rate of 0.2 and a max 300 estimators.

In [ ]:
hyper_params = {n_estimators = 300, learning_rate = 0.2, verbosity = 1, 
             random_state = RANDOM_STATE,  early_stopping_rounds=20, tree_method="hist"}
eval_set = [(X_cv, y_cv)]

## 1. Baseline / Control
We must first establish a baseline to compare the 3 techniques to. Hopefully they won't all just be worse than the baseline!

In [ ]:
xgb_model = XGBClassifier(hyper_params)
xgb_model.fit(X_train, y_train, eval_set = eval_set)

In [ ]:
preds = xgb_model.predict(X_test)
evaluate(preds, y_test, le.classes_, "baseline")

## 2. Sample weights
We will now use the XGBoost `sample_weight` parameter to weigh attack classes with a greater cost proportional to how rare they are. This can be automatically calculated with sklearn `compute_sample_weight` with "balanced" mode which adjusts weights inversely proportional to class frequencies. We will calculate our sample weights ONLY on the training set to avoid leakage.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)

In [ ]:
xgb_sw = XGBClassifier(hyper_params)
xgb_sw.fit(X_train, y_train, eval_set = eval_set, sample_weight=sample_weights)

In [ ]:
preds_sw = xgb_sw.predict(X_test)
evaluate(preds_sw, y_test, le.classes_, "sample weights")

## 3. Undersampling
There are two classes of undersampling algorithms, **generation** and **selection**. They both involve creating a new set S' from an original set S, where |S'| < |S|. However, in generation S' ⊄ S wheras in selection S' ⊂ S. 

For the sake of fairness, we will attempt to use a common under-sampling strategy. This will be to tune the majority classes down to have a frequency equal to the third quartile (median would remove far too many examples).

In [ ]:
from collections import Counter
counts = Counter(y_train)
vals = list(counts.values())
T_under = int(np.percentile(vals, 75))   # Q3
under_strategy = {c: T_under for c, n in counts.items() if n > T_under}

### 3.1 Generation
For generation we will use the cluster centroids method from imbalanced learn, which uses K-means reduce a majority class to the centroids of K-means.

In [ ]:
from imblearn.under_sampling import ClusterCentroids
cc = ClusterCentroids(sampling_strategy=under_strategy, random_state=RANDOM_STATE)
X_train_cc, y_train_cc = cc.fit_resample(X_train, y_train)

xgb_cc = XGBClassifier(hyper_params)
xgb_cc.fit(X_train_cc, y_train_cc, eval_set=eval_set)

preds_cc = xgb_cc.predict(X_test)
evaluate(preds_cc, y_test, le.classes_, "Cluster Centroids (under)"

### 3.2 Selection
Selection is split into 2 further groups (my laptop can't take it any more): Controlled an Cleaning. Controlled involves reducing majority classes down to an arbitrary user-specified amount (we will use third quartile as above), and cleaning involves using an algorithm to remove observations that follow a certain criteria, thus the amount of samples at the end can not be user-specified. 

#### 3.2.1 Controlled random
We will random under sampling as the controlled example.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=RANDOM_STATE)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

xgb_rus = XGBClassifier(hyper_params)
xgb_rus.fit(X_train_rus, y_train_rus, eval_set=eval_set)

preds_rus = xgb_rus.predict(X_test)
evaluate(preds_rus, y_test, le.classes_, "controlled random (under)")

#### 3.2.2 Cleaning ENN
I have decided to use edited nearest neighbours over Tomek Links due to it's higher aggression. For the purposes of time, I will ignore repeated and all KNN variations of ENN.

In [ ]:
from imblearn.under_sampling import EditedNearestNeighbours
enn = EditedNearestNeighbours(n_jobs=-1)
X_train_enn, y_train_enn = enn.fit_resample(X_train, y_train)

xgb_enn = XGBClassifier(hyper_params)
xgb_enn.fit(X_train_enn, y_train_enn, eval_set=eval_set)

preds_enn = xgb_enn.predict(X_test)
evaluate(preds_rnn, y_test, le.classes_, "ENN (under)")

## 4. Oversampling

We will try two methods of oversampling:
- Naive random,
- SMOTE.

These will both be using methods from the imbalanced learn library. For the sake of fairness, the strategy will be to bring each class to have a frequency equal to the median frequency (of the training set). So, if the median is 5,000, the classes with the least frequent half will be oversampled up to 5,000 entires.

In [ ]:
from collections import Counter
counts = Counter(y_train)
T = int(np.median(list(counts.values())))
k = 5
strategy = {c: T for c, n in counts.items() if k < n < T}

### 4.1 Naive Random
The simplest oversampling method would be naive random - sampling with replacement. This would allow rows of minority classes to be duplicated multiple times. While this could potentially help some rare classes (likely would not help classes with ~10 entries), it is functionally the same as changing the weight of the class from (2). 

Despite this, we will still trial the method.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=RANDOM_STATE, shrinkage=1, sampling_strategy=strategy)
X_train_r, y_train_r = ros.fit_resample(X_train, y_train)

xgb_ros = XGBClassifier(hyper_params)
xgb_ros.fit(X_train_r, y_train_r, eval_set = eval_set)

In [ ]:
preds_ros = xgb_ros.predict(X_test)
evaluate(preds_ros, y_test, le.classes_, "Naive oversampling (median)")

### 4.2 SMOTE - synthetic minority oversampling technique

Creates entries in rare classes by interpolating between 2 actually existing entries. For the sake of the test, I will allow very small classes to also be SMOTE'd. Even classes which have <10 entries. It will be interesting to see how the model changes when it recieves thousands of synthetic data points.

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(sampling_strategy=strategy, k_neighbors=k)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

In [ ]:
xgb_sm = XGBClassifier(hyper_params)
xgb_sm.fit(X_train_sm, y_train_sm, eval_set = eval_set)

In [ ]:
preds_sm = xgb_sm.predict(X_test)
evaluate(preds_sm, y_test, le.classes_, "SMOTE (median)")